# 1. YouTube Data Collection: Educational Channels Pipeline

## 1.1. Project Overview

## 1.2. Library Imports

In [2]:
import googleapiclient.discovery
import pandas as pd
import time

## 1.3. API Authentication & Configuration

In this section, we define the credentials required to communicate with Google's servers.

In [3]:
# --- Configuration ---

API_KEY = "YOUR_API_KEY_HERE"


# Initialize the YouTube Data API client (Version 3)
# This 'youtube' object will be used to make all subsequent API requests
youtube = googleapiclient.discovery.build(
    "youtube", 
    "v3", 
    developerKey=API_KEY
)

# The list of 90 domains

keywords = [
    # 
    "active learning strategies",
    "flipped classroom model",
    "project-based learning ideas",
    "differentiated instruction examples",
    "classroom management techniques",
    "formative assessment tools",
    "how to engage students",
    "teaching with technology",
    "lesson planning for teachers",
    "inquiry-based learning",
    "cooperative learning structures",
    "scaffolding in education",
    "universal design for learning",
    "gamification in education",
    "student-centered learning",

    # 
    "homeschooling curriculum reviews",
    "unschooling explained",
    "how to teach reading at home",
    "educational activities for toddlers",
    "positive discipline strategies",
    "how to help with homework",
    "Montessori at home",
    "Waldorf education philosophy",
    "co-op homeschooling groups",
    "virtual school vs homeschool",
    "social emotional learning at home",
    "screen time for kids",
    "raising bilingual children",
    "early childhood development milestones",
    "preparing child for kindergarten",

    # 
    "algebra for kids",
    "geometry proofs explained",
    "chemistry experiments at home",
    "biology animations",
    "physics demonstrations",
    "world history documentaries",
    "geography for beginners",
    "literature analysis tips",
    "creative writing prompts",
    "french grammar exercises",
    "mandarin chinese for beginners",
    "latin roots vocabulary",
    "art history timeline",
    "music theory basics",
    "coding for kids scratch",

    # 
    "autism teaching strategies",
    "dyslexia reading programs",
    "ADHD classroom accommodations",
    "speech therapy activities",
    "special education law",
    "IEP meeting tips",
    "assistive technology tools",
    "sensory processing disorder",
    "gifted education resources",
    "inclusive classroom practices",

    #
    "best educational apps for kids",
    "google classroom tutorial",
    "how to use kahoot",
    "quizlet flashcards tips",
    "interactive whiteboard lessons",
    "digital storytelling tools",
    "virtual reality in education",
    "coding platforms for students",
    "open educational resources",
    "learning management systems comparison",

    # 
    "college admission essay examples",
    "scholarships for international students",
    "how to choose a major",
    "gap year programs",
    "vocational training options",
    "resume for high school students",
    "career aptitude test",
    "internship opportunities",
    "study abroad programs",
    "financial aid application guide",

    # 
    "microschools explained",
    "democratic schools",
    "forest school activities",
    "sudbury model",
    "online charter schools",
    "hybrid homeschooling",
    "experiential learning",
    "outdoor education benefits",
    "project-based homeschool",
    "self-directed education",

    # 
    "teacher burnout prevention",
    "reflective teaching practice",
    "teacher portfolio examples",
    "professional learning communities",
    "educational leadership skills"
]

## 1.4. Automated Channel Discovery
The following function performs a massive search across YouTube. It filters for the **Education category (ID 27)** and handles API quotas to prevent script interruption.

In [4]:
def fetch_massive_channel_ids(keywords, target_per_keyword=50):
    # Initialize data structures for storage and duplicate tracking
    all_data = []
    seen_channels = set()
    stop_all = False
    
    # Iterate through the list of 90 domains
    for domain in keywords:
        if stop_all:
            break
        
        print(f"Searching for: {domain}...")
        
        try:
            # Call the YouTube Search API
            # Each request costs 100 units of the daily quota
            request = youtube.search().list(
                part="snippet",
                q=domain,
                type="video",
                videoCategoryId="27", # Filter for Educational content
                maxResults=target_per_keyword,
                relevanceLanguage="en" # Target English-speaking channels
            )
            response = request.execute()
            
            # Process each item in the search results
            for item in response.get('items', []):
                channel_id = item['snippet']['channelId']
                
                # Check for duplicates before adding to the list
                if channel_id not in seen_channels:
                    seen_channels.add(channel_id)

                    all_data.append({
                        "keyword": domain,
                        "channel_name": item['snippet']['title'],
                        "channel_id": item['snippet']['channelId']
                    })
                
                # Stop if the total collected channels reach 6,000
                if len(all_data) >= 6000:
                    stop_all = True
                    break
            
            # Short sleep to prevent hitting API rate limits (pacing)
            time.sleep(0.2)
            
        except Exception as e:
            # Handle API Quota exhaustion specifically
            if "quotaExceeded" in str(e):
                print("!!! Quota Exceeded. Stopping script.")
                stop_all = True
                break
            print(f"Error searching for {domain}: {e}")
            
    return all_data

## 1.5. Execution and Data Persistence

### Processing Results
After the search is complete, we convert the raw data into a structured format, remove any remaining duplicates, and save the final list to a CSV file.

In [5]:
# Execute the search function
collected_channels = fetch_massive_channel_ids(keywords, 50)

# Convert the results list into a Pandas DataFrame
df_channels = pd.DataFrame(collected_channels)

# Ensure no duplicate channel IDs exist in the final DataFrame
df_channels = df_channels.drop_duplicates(subset='channel_id')

# Export the collected data to a CSV file
df_channels.to_csv("eucation_channel_list.csv", index=False)
print(f"Done! Successfully collected {len(df_channels)} unique channels.")

# Preview the top rows of the result
df_channels.head()

Searching for: active learning strategies...
Searching for: flipped classroom model...
Searching for: project-based learning ideas...
Searching for: differentiated instruction examples...
Searching for: classroom management techniques...
Searching for: formative assessment tools...
Searching for: how to engage students...
Searching for: teaching with technology...
Searching for: lesson planning for teachers...
Searching for: inquiry-based learning...
Searching for: cooperative learning structures...
Searching for: scaffolding in education...
Searching for: universal design for learning...
Searching for: gamification in education...
Searching for: student-centered learning...
Searching for: homeschooling curriculum reviews...
Searching for: unschooling explained...
Searching for: how to teach reading at home...
Searching for: educational activities for toddlers...
Searching for: positive discipline strategies...
Searching for: how to help with homework...
Searching for: Montessori at ho

,keyword,channel_name,channel_id
0,active learning strategies,The Active Learning Method,UC-RKpEc4eE9PwJaupN91xYQ
1,active learning strategies,Active Learning Strategies,UCYSSg9rr-pgtK5UqkZdE_KQ
2,active learning strategies,5 Reasons You’re Doing Active Learning WRONG,UC3VWbWk8qDBiF0741izgpQg
3,active learning strategies,Active Learning Overview,UCEBb1b_L6zDS3xTUrIALZOw
4,active learning strategies,5 Proven Active Learning Strategies That Impro...,UC1uiV9laQtQEHzTj3TJLuEQ
